# Cambridge Dictionary Dataset: Professional Data Engineering & Quality Analysis
## An End-to-End Database Profile, Quality Audit, and Dimensional Modeling Strategy

### Author: Lead Data Engineer
---

### Executive Summary & Project Context
This notebook presents a comprehensive, professional-grade analysis of `cambridge.db`—a SQLite database containing crawled lexicographical data from the Cambridge Dictionary. 

As a **Data Engineer**, the goal is to evaluate this database through multiple lenses:
1. **System & DB Metadata**: Understand the storage footprints, physical SQLite settings, and table structures.
2. **Ingestion & Data Quality Audit**: Profile crawl execution, identify failure patterns, and chart data ingestion velocity.
3. **Lexicographical Profiling**: Categorize entries, analyze parts-of-speech distributions, and profile CEFR language proficiency levels.
4. **Relational Density & Semantic Graph Analysis**: Quantify structural relationships, such as senses per entry, example sentence distributions, and vocabulary-topic mappings.
5. **Production OLAP Warehousing Strategy**: Audit index structures, verify referential integrity, and propose a production-grade Star Schema and ELT architecture.

---


In [1]:
# Import analytical and visualization packages
import os
import sqlite3
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from contextlib import contextmanager

# Set Plotly theme to dark and notebook rendering
pio.templates.default = "plotly_dark"
pio.renderers.default = "plotly_mimetype"

DB_PATH = "cambridge.db"

@contextmanager
def get_connection():
    """Establishes a connection to the SQLite database and ensures it is closed on exit."""
    conn = sqlite3.connect(DB_PATH)
    try:
        yield conn
    finally:
        conn.close()

print(f"Connection helper initialized. Database file size: {os.path.getsize(DB_PATH) / (1024*1024):.2f} MB")


Connection helper initialized. Database file size: 111.99 MB


## Section 1: System & Database Metadata
Before looking at the data content, we must inspect the database's physical storage properties and configuration. We query internal SQLite parameters (PRAGMAs) and measure the scale of the dataset across all tables.


In [2]:
# Retrieve SQLite engine information and physical settings
with get_connection() as conn:
    sqlite_version = conn.execute("SELECT sqlite_version();").fetchone()[0]
    page_size = conn.execute("PRAGMA page_size;").fetchone()[0]
    page_count = conn.execute("PRAGMA page_count;").fetchone()[0]
    journal_mode = conn.execute("PRAGMA journal_mode;").fetchone()[0]
    synchronous = conn.execute("PRAGMA synchronous;").fetchone()[0]

db_size_bytes = os.path.getsize(DB_PATH)
db_size_mb = db_size_bytes / (1024 * 1024)

print(f"=== SQLite System Configuration ===")
print(f"SQLite Version : {sqlite_version}")
print(f"Database File  : {DB_PATH} ({db_size_mb:.2f} MB)")
print(f"Page Size      : {page_size} bytes")
print(f"Page Count     : {page_count}")
print(f"Journal Mode   : {journal_mode.upper()} (WAL = Write-Ahead Log)")
print(f"Synchronous    : {synchronous} (2 = FULL durability)")


=== SQLite System Configuration ===
SQLite Version : 3.41.2
Database File  : cambridge.db (111.99 MB)
Page Size      : 4096 bytes
Page Count     : 28729
Journal Mode   : WAL (WAL = Write-Ahead Log)
Synchronous    : 2 (2 = FULL durability)


In [3]:
# Quantify the scale of the dataset by counting records in each table
tables = [
    "words", "entries", "entry_inflections", "senses", 
    "sense_synonyms", "examples", "collocations", "topics", "word_topics"
]

row_counts = []
with get_connection() as conn:
    for table in tables:
        count = conn.execute(f"SELECT COUNT(*) FROM {table};").fetchone()[0]
        row_counts.append({"Table": table, "RowCount": count})

df_counts = pd.DataFrame(row_counts).sort_values(by="RowCount", ascending=False)

# Visualize Table row counts
fig_counts = px.bar(
    df_counts,
    x="RowCount",
    y="Table",
    orientation="h",
    text_auto=True,
    title="Cambridge.db Dataset Scale: Row Counts by Table",
    labels={"RowCount": "Record Count", "Table": "Database Table"},
    color="RowCount",
    color_continuous_scale="Viridis"
)
fig_counts.update_layout(yaxis={'categoryorder':'total ascending'}, height=450, margin=dict(l=150, r=20, t=50, b=50))
fig_counts.show()


## Section 2: Ingestion & Data Quality Audit
An essential step for a data engineer is auditing the data ingestion pipeline. In this section, we analyze:
* **Crawl Success Rates**: The status of scraped URL slugs.
* **Error Message Patterns**: Categorization of crawl-time errors.
* **Ingestion Velocity**: The rate at which the database was populated (words crawled per 10-minute interval).


In [4]:
# Analyze crawl status distribution
with get_connection() as conn:
    df_status = pd.read_sql_query(
        "SELECT status, COUNT(*) as count FROM words GROUP BY status;", conn
    )

df_status["percentage"] = (df_status["count"] / df_status["count"].sum() * 100).round(2)
print("=== Ingestion Crawl Status ===")
print(df_status.to_string(index=False))

# Pie Chart
fig_status = px.pie(
    df_status,
    values="count",
    names="status",
    title="Vocabulary Ingestion Status Distribution",
    hole=0.4,
    color_discrete_sequence=px.colors.qualitative.Pastel
)
fig_status.update_traces(textinfo="percent+label")
fig_status.show()


=== Ingestion Crawl Status ===
   status  count  percentage
     done  87461       77.34
not_found   3621        3.20
  pending  21959       19.42
     seen     45        0.04


In [5]:
# Profile and classify ingestion errors
with get_connection() as conn:
    df_errors = pd.read_sql_query(
        "SELECT error_msg, COUNT(*) as count FROM words WHERE error_msg IS NOT NULL GROUP BY error_msg ORDER BY count DESC;", 
        conn
    )

print("=== Ingestion Errors Summary ===")
print(df_errors.to_string(index=False))

if not df_errors.empty:
    fig_errors = px.bar(
        df_errors,
        x="count",
        y="error_msg",
        orientation="h",
        text_auto=True,
        title="Ingestion Error Classification & Frequencies",
        labels={"count": "Failure Count", "error_msg": "Error Message"},
        color="count",
        color_continuous_scale="Reds"
    )
    fig_errors.update_layout(yaxis={'categoryorder':'total ascending'}, height=300)
    fig_errors.show()


=== Ingestion Errors Summary ===
           error_msg  count
    No entries found   3621
HTTP Error status: 0      5


In [6]:
# Analyze crawl velocity by partitioning crawled timestamps into 10-minute buckets
with get_connection() as conn:
    # SQLite SQL query to group dates by 10-minute buckets
    df_velocity = pd.read_sql_query("""
        SELECT 
            SUBSTR(crawled_at, 1, 10) || ' ' || SUBSTR(crawled_at, 12, 4) || '0' as time_bucket,
            COUNT(*) as words_crawled
        FROM words
        WHERE crawled_at IS NOT NULL
        GROUP BY time_bucket
        ORDER BY time_bucket;
    """, conn)

df_velocity['time_bucket'] = pd.to_datetime(df_velocity['time_bucket'])

# Line Chart of Ingestion velocity
fig_velocity = px.line(
    df_velocity,
    x="time_bucket",
    y="words_crawled",
    title="Data Ingestion Velocity (Words Crawled per 10-Minute Window)",
    labels={"time_bucket": "Time Bucket (10m)", "words_crawled": "Words Crawled"},
    markers=True
)
fig_velocity.update_traces(line_color="#00CC96", line_width=2)
fig_velocity.update_layout(height=450)
fig_velocity.show()


## Section 3: Lexicographical & Dimensional Profiling
Now we turn to the content of the dictionary, modeling various vocabulary dimensions. We analyze the linguistic types of words, parts of speech, dictionary sources, and CEFR language proficiency levels.


In [7]:
# Query distributions for word type categories and part of speech tags
with get_connection() as conn:
    df_entry_type = pd.read_sql_query(
        "SELECT entry_type, COUNT(*) as count FROM words GROUP BY entry_type ORDER BY count DESC;", conn
    )
    df_pos = pd.read_sql_query(
        "SELECT pos, COUNT(*) as count FROM entries GROUP BY pos ORDER BY count DESC LIMIT 15;", conn
    )

# Clean empty POS values
df_pos['pos'] = df_pos['pos'].replace('', 'Unknown/Unclassified')

# Plot Entry Types
fig_entry_type = px.bar(
    df_entry_type,
    x="entry_type",
    y="count",
    title="Vocabulary Distribution by Entry Type Category",
    labels={"count": "Count", "entry_type": "Entry Type"},
    color="entry_type",
    color_discrete_sequence=px.colors.qualitative.Bold
)
fig_entry_type.show()

# Plot POS
fig_pos = px.bar(
    df_pos,
    x="count",
    y="pos",
    orientation="h",
    text_auto=True,
    title="Top 15 Parts of Speech (POS) Distribution",
    labels={"count": "Entry Count", "pos": "Part of Speech"},
    color="count",
    color_continuous_scale="Cividis"
)
fig_pos.update_layout(yaxis={'categoryorder':'total ascending'}, height=500)
fig_pos.show()


In [8]:
# Profile CEFR Language Proficiency levels
with get_connection() as conn:
    df_cefr = pd.read_sql_query(
        "SELECT cefr_level, COUNT(*) as count FROM senses GROUP BY cefr_level ORDER BY cefr_level;", conn
    )

df_cefr['cefr_level'] = df_cefr['cefr_level'].replace('', 'Not Annotated')

# Order CEFR levels logically from beginner (A1) to proficient (C2)
cefr_order = ["A1", "A2", "B1", "B2", "C1", "C2", "Not Annotated"]
df_cefr['cefr_level'] = pd.Categorical(df_cefr['cefr_level'], categories=cefr_order, ordered=True)
df_cefr = df_cefr.sort_values(by="cefr_level")

# Show raw distribution
print("=== CEFR Levels Distribution ===")
print(df_cefr.to_string(index=False))

# Plot all including Not Annotated
fig_cefr = px.bar(
    df_cefr,
    x="cefr_level",
    y="count",
    title="Senses Distribution across CEFR Levels (Full Dataset)",
    labels={"count": "Sense Count", "cefr_level": "CEFR Level"},
    color="cefr_level",
    color_discrete_map={
        "A1": "#1a9641", "A2": "#a6d96a", "B1": "#fdae61", "B2": "#f46d43",
        "C1": "#d73027", "C2": "#a50026", "Not Annotated": "#737373"
    }
)
fig_cefr.show()

# Clean visual: Graded vocabulary distribution only (A1-C2)
df_cefr_annotated = df_cefr[df_cefr['cefr_level'] != 'Not Annotated'].copy()
fig_cefr_annotated = px.bar(
    df_cefr_annotated,
    x="cefr_level",
    y="count",
    title="CEFR Level Distribution for Graded Vocabulary (A1 - C2)",
    labels={"count": "Sense Count", "cefr_level": "CEFR Level"},
    color="cefr_level",
    color_discrete_sequence=px.colors.sequential.Sunset_r
)
fig_cefr_annotated.show()


=== CEFR Levels Distribution ===
   cefr_level  count
           A1    646
           A2   1366
           B1   2630
           B2   3692
           C1   2047
           C2   3022
Not Annotated 159987


In [9]:
# Categorize raw dictionary source values into normalized dimensions
with get_connection() as conn:
    df_sources = pd.read_sql_query("""
        SELECT 
          CASE 
            WHEN dictionary_source LIKE '%Learner%' THEN 'Cambridge Advanced Learner''s Dictionary'
            WHEN dictionary_source LIKE '%American%' THEN 'American Dictionary'
            WHEN dictionary_source LIKE '%Business%' THEN 'Business English'
            ELSE 'Other/Custom'
          END as source_category,
          COUNT(*) as count
        FROM entries
        GROUP BY source_category;
    """, conn)

# Plot Sources
fig_sources = px.pie(
    df_sources,
    values="count",
    names="source_category",
    title="Data Source Distribution by Source Category",
    hole=0.4,
    color_discrete_sequence=px.colors.qualitative.Set2
)
fig_sources.update_traces(textinfo="percent+label")
fig_sources.show()


In [10]:
# Profile grammatical attributes (extract countable/uncountable nouns & transitive/intransitive verbs)
with get_connection() as conn:
    df_grammar = pd.read_sql_query("""
        SELECT grammar, COUNT(*) as count 
        FROM entries 
        WHERE grammar IS NOT NULL AND grammar != '' 
        GROUP BY grammar 
        ORDER BY count DESC 
        LIMIT 12;
    """, conn)

# Visualizing grammatical counts
fig_grammar = px.bar(
    df_grammar,
    x="count",
    y="grammar",
    orientation="h",
    text_auto=True,
    title="Top 12 Grammatical Annotations Frequency",
    labels={"count": "Frequency", "grammar": "Grammar Label"},
    color="count",
    color_continuous_scale="Mint"
)
fig_grammar.update_layout(yaxis={'categoryorder':'total ascending'})
fig_grammar.show()


## Section 4: Relational Graph & Semantic Density Analysis
Here, we profile database entity connections. In data engineering, assessing relational density (cardinality distributions) is critical for database query design, indexing, and sizing. We evaluate:
1. **Senses per Entry** (Polysemy rate).
2. **Examples per Sense** (Contextual support).
3. **Semantic Relations** (Synonyms vs Antonyms count).
4. **Topics per Word** (Topic categorization density).


In [11]:
# Query density distributions (cardinality counts)
with get_connection() as conn:
    df_senses_per_entry = pd.read_sql_query(
        "SELECT entry_id, COUNT(*) as sense_count FROM senses GROUP BY entry_id;", conn
    )
    df_examples_per_sense = pd.read_sql_query(
        "SELECT sense_id, COUNT(*) as example_count FROM examples GROUP BY sense_id;", conn
    )

print("=== Descriptive Stats: Senses per Entry ===")
print(df_senses_per_entry["sense_count"].describe())
print("\n=== Descriptive Stats: Examples per Sense ===")
print(df_examples_per_sense["example_count"].describe())

# Histogram of Senses per Entry (Log Scale due to exponential falloff)
fig_senses_hist = px.histogram(
    df_senses_per_entry,
    x="sense_count",
    title="Senses per Entry Cardinality Distribution (Polysemy Degree)",
    labels={"sense_count": "Senses Count", "count": "Frequency"},
    log_y=True,
    color_discrete_sequence=["#AB63FA"]
)
fig_senses_hist.update_layout(height=400)
fig_senses_hist.show()

# Histogram of Examples per Sense (Log Scale)
fig_examples_hist = px.histogram(
    df_examples_per_sense,
    x="example_count",
    title="Examples per Sense Cardinality Distribution (Context Density)",
    labels={"example_count": "Examples Count", "count": "Frequency"},
    log_y=True,
    color_discrete_sequence=["#FFA15A"]
)
fig_examples_hist.update_layout(height=400)
fig_examples_hist.show()


=== Descriptive Stats: Senses per Entry ===
count    128099.000000
mean          1.353562
std           1.001162
min           1.000000
25%           1.000000
50%           1.000000
75%           1.000000
max          32.000000
Name: sense_count, dtype: float64

=== Descriptive Stats: Examples per Sense ===
count    135676.000000
mean          3.134946
std           2.432057
min           1.000000
25%           1.000000
50%           2.000000
75%           5.000000
max          32.000000
Name: example_count, dtype: float64


In [12]:
# Compare synonyms vs antonyms in the database
with get_connection() as conn:
    df_syn = pd.read_sql_query(
        "SELECT is_antonym, COUNT(*) as count FROM sense_synonyms GROUP BY is_antonym;", conn
    )

df_syn["relation_type"] = df_syn["is_antonym"].map({0: "Synonym", 1: "Antonym"})

fig_syn = px.bar(
    df_syn,
    x="relation_type",
    y="count",
    title="Semantic Relation Coverage: Synonyms vs Antonyms",
    labels={"count": "Relation Count", "relation_type": "Relation Type"},
    color="relation_type",
    color_discrete_sequence=["#00CC96", "#EF553B"]
)
fig_syn.update_layout(width=500, height=400)
fig_syn.show()


In [13]:
# Find top 20 topics associated with vocabulary
with get_connection() as conn:
    df_top_topics = pd.read_sql_query("""
        SELECT t.title, COUNT(wt.word_id) as word_count 
        FROM word_topics wt
        JOIN topics t ON wt.topic_id = t.id
        GROUP BY t.id
        ORDER BY word_count DESC
        LIMIT 20;
    """, conn)

fig_topics = px.bar(
    df_top_topics,
    x="word_count",
    y="title",
    orientation="h",
    text_auto=True,
    title="Top 20 Dictionary Topics by Word Association",
    labels={"word_count": "Words Associated", "title": "Topic Title"},
    color="word_count",
    color_continuous_scale="Turbo"
)
fig_topics.update_layout(yaxis={'categoryorder':'total ascending'}, height=550)
fig_topics.show()


## Section 5: Data Engineering & Schema Optimization Strategy
As a professional Data Engineer, checking schema health, referential constraints, indexing coverage, and sketching scalable production architectures is crucial.

### 1. Database Index Audit
Indexes improve join speeds between primary-foreign keys. We inspect active database indexes:


In [14]:
# Inspect active indexes in SQLite
with get_connection() as conn:
    df_indexes = pd.read_sql_query(
        "SELECT tbl_name as table_name, name as index_name, sql as ddl FROM sqlite_master WHERE type='index';", 
        conn
    )

print("=== Existing Indexes in SQLite Schema ===")
for idx, row in df_indexes.iterrows():
    print(f"Table: {row['table_name']:<18} | Index: {row['index_name']:<22}")
    print(f" DDL: {row['ddl']}")
    print("-" * 80)


=== Existing Indexes in SQLite Schema ===
Table: words              | Index: sqlite_autoindex_words_1
 DDL: None
--------------------------------------------------------------------------------
Table: entry_inflections  | Index: idx_inflections_form  
 DDL: CREATE INDEX idx_inflections_form ON entry_inflections(inflected_form)
--------------------------------------------------------------------------------
Table: topics             | Index: sqlite_autoindex_topics_1
 DDL: None
--------------------------------------------------------------------------------
Table: word_topics        | Index: sqlite_autoindex_word_topics_1
 DDL: None
--------------------------------------------------------------------------------
Table: words              | Index: idx_words_status      
 DDL: CREATE INDEX idx_words_status   ON words(status)
--------------------------------------------------------------------------------
Table: entries            | Index: idx_entries_word      
 DDL: CREATE INDEX idx_entr

In [15]:
# Validate referential integrity constraints (checking for orphan records)
integrity_queries = {
    "Orphaned Senses (senses without entries)": 
        "SELECT COUNT(*) FROM senses WHERE entry_id NOT IN (SELECT id FROM entries);",
    "Orphaned Entries (entries without words)": 
        "SELECT COUNT(*) FROM entries WHERE word_id NOT IN (SELECT id FROM words);",
    "Orphaned Examples (examples without senses)": 
        "SELECT COUNT(*) FROM examples WHERE sense_id NOT IN (SELECT id FROM senses);",
    "Orphaned Synonyms (synonyms without senses)": 
        "SELECT COUNT(*) FROM sense_synonyms WHERE sense_id NOT IN (SELECT id FROM senses);",
    "Orphaned Inflections (inflections without entries)": 
        "SELECT COUNT(*) FROM entry_inflections WHERE entry_id NOT IN (SELECT id FROM entries);",
    "Orphaned Word Topics (links without valid words/topics)": 
        "SELECT COUNT(*) FROM word_topics WHERE word_id NOT IN (SELECT id FROM words) OR topic_id NOT IN (SELECT id FROM topics);"
}

audit_rows = []
with get_connection() as conn:
    for check_title, sql in integrity_queries.items():
        count = conn.execute(sql).fetchone()[0]
        audit_rows.append({
            "Referential Integrity Check": check_title, 
            "Orphan Count": count, 
            "Status": "PASS" if count == 0 else "FAIL"
        })

df_audit = pd.DataFrame(audit_rows)
print("=== Referential Integrity Audit Results ===")
print(df_audit.to_string(index=False))


=== Referential Integrity Audit Results ===
                            Referential Integrity Check  Orphan Count Status
               Orphaned Senses (senses without entries)             0   PASS
               Orphaned Entries (entries without words)             0   PASS
            Orphaned Examples (examples without senses)             0   PASS
            Orphaned Synonyms (synonyms without senses)             0   PASS
     Orphaned Inflections (inflections without entries)             0   PASS
Orphaned Word Topics (links without valid words/topics)             0   PASS


### 2. Architectural Recommendations: Production OLAP Warehousing

The SQLite database uses a highly normalized schema (3rd Normal Form - 3NF). While this is optimized for scraping concurrency and writes (OLTP), it results in slow query performances for large analytics (due to joining up to 5-7 tables). 

For enterprise-level business intelligence, data science, and search engines, we recommend transitioning to a **Star Schema** on a modern Cloud Data Warehouse (e.g., Snowflake, BigQuery, or Databricks).

#### Proposed Dimensional Model (Star Schema)
```
          +----------------------+
          |   dim_calendar       |
          +----------------------+
          | * date_key (PK)      |
          |   date               |
          |   year, month, day   |
          +----------+-----------+
                     |
                     | 1:N
                     v
+------------------+   N:1   +----------------------+
| dim_words        |<--------+  fact_senses         |
+------------------+         +----------------------+
| * word_key (PK)  |         | * sense_key (PK)     |
|   word_slug      |         | * word_key (FK)      |
|   display_form   |         | * entry_key (FK)     |
|   entry_type     |         | * date_key (FK)      |
|   status         |         |   cefr_level         |
+------------------+         |   definition         |
                             |   guideword          |
                             |   grammar_pattern    |
                             |   domain             |
+------------------+         |   is_annotated_cefr  |
| dim_entries      |<--------+                      |
+------------------+   N:1   +----------+-----------+
| * entry_key (PK) |                    |
|   headword       |                    | 1:N
|   pos            |                    v
|   pronunciation  |         +----------------------+
|   grammar        |         |  fact_examples       |
|   dict_source    |         +----------------------+
+------------------+         | * example_key (PK)   |
                             | * sense_key (FK)     |
                             |   example_sentence   |
                             |   is_extra           |
                             |   collocation        |
                             +----------------------+
```

#### Production ETL Pipeline Architecture
1. **Extraction (EL)**:
   - Run the scraper orchestrating using **Apache Airflow** or **Prefect**.
   - Output files (SQLite files or CSV/Parquet chunks) are exported to cloud storage buckets (Amazon S3 / Google Cloud Storage).
2. **Loading (L)**:
   - Load raw tables directly into staging schemas in Snowflake / BigQuery / Databricks Delta Lake.
3. **Transformation (T)**:
   - Use **dbt (data build tool)** to run batch SQL transformation models.
   - Standardize POS strings, categorize dictionary sources, parse JSON arrays in `senses.labels` into child relational fields, and create the optimized star schema fact and dimension tables.
   - Implement data quality tests (using `dbt test` or Great Expectations) to enforce uniqueness, non-nullability, and referential integrity constraints on dimensions.
